## Import библиотек и модулей


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os, sys
import json

from pathlib import Path
import re
from datetime import datetime, timedelta
# --- Настройка путей и sys.path ---
# Добавляем корневую директорию проекта в sys.path для импорта кастомных модулей
PROJECT_ROOT = Path().cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


from src.config import path_config, database
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from concurrent.futures import ThreadPoolExecutor, as_completed
from sqlalchemy import text
import os, sys
import re
from datetime import datetime, timedelta
# --- Настройка путей и sys.path ---
# Добавляем корневую директорию проекта в sys.path для импорта кастомных модулей
PROJECT_ROOT = Path().cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


from src.logger import logger
from src.database import clickhouse_engine

2026-03-18 15:54:22,417 | my_logger - INFO - ✅ ClickHouse engine создан | /data/aturov/universal_control_group/src/database.py:21


## Constants


In [2]:
DATA_START = "2026-02-01"
OUTPUT_DIR = path_config.data_raw_path
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

params = {"DATA_START": DATA_START}
output_path_1 = OUTPUT_DIR / f"ukg_eligible_base_{DATA_START}.parquet"
output_path_1

PosixPath('/data/aturov/universal_control_group/data/raw/ukg_eligible_base_2026-02-01.parquet')

## Dowload

In [3]:
import clickhouse_connect

def get_client():
    return clickhouse_connect.get_client(
        host=os.environ["CLICKHOUSE_HOST"],
        port=int(os.environ["CLICKHOUSE_PORT"]),
        username=os.environ["CLICKHOUSE_USER"],
        password=os.environ["CLICKHOUSE_PASSWORD"],
        database="DWH",
        secure=os.environ.get("CLICKHOUSE_PROTOCOL", "https") == "https",
        verify=False,
    )
client = get_client()

In [4]:

sql_path = Path(f"{PROJECT_ROOT}/sql/01_ukg_eligible_base.sql")
sql_text = sql_path.read_text(encoding="utf-8")

query = (
    sql_text
    .replace(":DATA_START", f"'{DATA_START}'")
)

output_path = Path(f"{PROJECT_ROOT}/data/raw/ukg_eligible_base_{DATA_START}.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

writer = None
total_rows = 0

try:
    with client.query_arrow_stream(query) as stream:
        for batch in stream:
            if writer is None:
                writer = pq.ParquetWriter(output_path, batch.schema, compression="snappy")
            writer.write_batch(batch)
            total_rows += batch.num_rows
            logger.info(f"wrote {total_rows} rows")
finally:
    if writer is not None:
        writer.close()

logger.info(f"Data saved to {output_path}")


2026-03-18 15:54:22,757 | my_logger - INFO - wrote 8188 rows | /tmp/ipykernel_1128249/1443850864.py:22
2026-03-18 15:54:22,802 | my_logger - INFO - wrote 16379 rows | /tmp/ipykernel_1128249/1443850864.py:22
2026-03-18 15:54:22,848 | my_logger - INFO - wrote 24538 rows | /tmp/ipykernel_1128249/1443850864.py:22
2026-03-18 15:54:22,890 | my_logger - INFO - wrote 32704 rows | /tmp/ipykernel_1128249/1443850864.py:22
2026-03-18 15:54:22,939 | my_logger - INFO - wrote 40876 rows | /tmp/ipykernel_1128249/1443850864.py:22
2026-03-18 15:54:22,964 | my_logger - INFO - wrote 49067 rows | /tmp/ipykernel_1128249/1443850864.py:22
2026-03-18 15:54:23,016 | my_logger - INFO - wrote 57256 rows | /tmp/ipykernel_1128249/1443850864.py:22
2026-03-18 15:54:23,068 | my_logger - INFO - wrote 65438 rows | /tmp/ipykernel_1128249/1443850864.py:22
2026-03-18 15:54:23,120 | my_logger - INFO - wrote 73626 rows | /tmp/ipykernel_1128249/1443850864.py:22
2026-03-18 15:54:23,139 | my_logger - INFO - wrote 81817 rows | /

In [5]:
sql_text_2 = (
    Path(f"{PROJECT_ROOT}/sql/02_hfct_subs_eligible_ids.sql")
    .read_text(encoding="utf-8")
)

sql_2 = text(sql_text_2)

output_path_2 = Path(f"{PROJECT_ROOT}/data/raw/hfct_subs_eligible_ids_{DATA_START}.parquet")
output_path_2.parent.mkdir(parents=True, exist_ok=True)

writer = None
total_rows = 0

try:
    with clickhouse_engine.connect() as conn:
        chunks = pd.read_sql(sql_2, conn, params=params, chunksize=100_000)

        for chunk in chunks:
            table = pa.Table.from_pandas(chunk, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(output_path_2, table.schema, compression="snappy")

            writer.write_table(table)
            total_rows += len(chunk)
            logger.info(f"wrote {total_rows} rows")
finally:
    if writer is not None:
        writer.close()

logger.info(f"Data saved to {output_path_2}")

2026-03-18 15:54:43,381 | my_logger - INFO - wrote 100000 rows | /tmp/ipykernel_1128249/1719658404.py:26
2026-03-18 15:54:43,626 | my_logger - INFO - wrote 200000 rows | /tmp/ipykernel_1128249/1719658404.py:26
2026-03-18 15:54:43,869 | my_logger - INFO - wrote 300000 rows | /tmp/ipykernel_1128249/1719658404.py:26
2026-03-18 15:54:44,100 | my_logger - INFO - wrote 400000 rows | /tmp/ipykernel_1128249/1719658404.py:26
2026-03-18 15:54:44,323 | my_logger - INFO - wrote 500000 rows | /tmp/ipykernel_1128249/1719658404.py:26
2026-03-18 15:54:45,025 | my_logger - INFO - wrote 600000 rows | /tmp/ipykernel_1128249/1719658404.py:26
2026-03-18 15:54:45,235 | my_logger - INFO - wrote 700000 rows | /tmp/ipykernel_1128249/1719658404.py:26
2026-03-18 15:54:45,440 | my_logger - INFO - wrote 800000 rows | /tmp/ipykernel_1128249/1719658404.py:26
2026-03-18 15:54:45,637 | my_logger - INFO - wrote 900000 rows | /tmp/ipykernel_1128249/1719658404.py:26
2026-03-18 15:54:45,828 | my_logger - INFO - wrote 1000

In [6]:
df_2 = pd.read_parquet(output_path_2)
df_1 = pd.read_parquet(output_path_1)
df_1.shape, df_2.shape, df_1.dtypes, df_2.dtypes
df_2.head(), df_1.head()

(   subscription_id
 0          9062542
 1          9062716
 2          9062719
 3          9062827
 4          9062902,
       DT           CTN   SUBS_ID  CUST_LEVEL     STATUS     REGION_CELL  \
 0  20485  996220508564  22503628  B2B_CREDIT     Active          Бишкек   
 1  20485  996220508565  22503640  B2B_CREDIT     Active          Бишкек   
 2  20485  996220508567  22500424  B2B_CREDIT     Active  Ошская область   
 3  20485  996220508573  22500490  B2B_CREDIT     Active  Ошская область   
 4  20485  996220508574  22591918  B2B_HYBRID  Suspended                   
 
   PRICE_PLAN PRICE_PLAN_RU PERIODICITY  SUBSCRIPTION_FEE  ... FLAG_ABONKA  \
 0      MODEM         Модем                           0.0  ...           0   
 1      MODEM         Модем                           0.0  ...           0   
 2      MODEM         Модем                           0.0  ...           0   
 3      MODEM         Модем                           0.0  ...           0   
 4     M2M100  M2M_100_25GB    

In [7]:
eligible_ids = set(df_2["subscription_id"])
df_final = df_1[df_1["SUBS_ID"].isin(eligible_ids)].copy()

logger.info(f"Final filtered dataset rows: {len(df_final)}")

2026-03-18 15:54:58,031 | my_logger - INFO - Final filtered dataset rows: 2921367 | /tmp/ipykernel_1128249/3743193958.py:4


In [8]:
df_final['FLAG_4G'].dtypes

dtype('uint8')

In [14]:
df_final.dtypes.head(100)

DT                uint16
CTN               object
SUBS_ID           uint64
CUST_LEVEL        object
STATUS            object
                   ...  
MULTIPLAY          uint8
LIFETIME_TOTAL     int32
M2M_FLAG           uint8
GENDER            object
AGE               object
Length: 97, dtype: object

In [15]:
df_final

,DT,CTN,SUBS_ID,CUST_LEVEL,STATUS,REGION_CELL,PRICE_PLAN,PRICE_PLAN_RU,PERIODICITY,SUBSCRIPTION_FEE,...,FLAG_ABONKA,TOTAL_MOU,FIRST_SIM,MY_BEELINE_USER,BALANCE_USER,MULTIPLAY,LIFETIME_TOTAL,M2M_FLAG,GENDER,AGE
4,20485,996220508574,22591918,B2B_HYBRID,Suspended,,M2M100,M2M_100_25GB,1 months,100.0,...,0,0.000000,0,0,0,0,109,0,,
9,20485,996220508585,19379818,B2C,Active,,SUPERXMAX,Безлимит 500,30 days,500.0,...,0,0.000000,0,0,0,0,409,0,male,26 - 35 years
15,20485,996220508636,672963,B2C,Active,Ошская область,BIRGEINTERNETIVI,Бирге 2.0,28 days,540.0,...,1,158.483333,1,0,0,0,2223,0,female,26 - 35 years
25,20485,996220508666,20947156,B2C,Active,Бишкек,FAMILY,Семейный,,0.0,...,0,0.000000,1,0,0,0,141,0,female,36 - 45 years
34,20485,996220508686,2576656,B2C,Active,Бишкек,SUPERXMAX,Безлимит 500,30 days,500.0,...,0,45.333333,0,0,0,0,2025,0,male,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3244687,20485,996220507775,1116688,B2C,Active,,CVMSWITCH220,CVM Включай 220,30 days,220.0,...,0,0.700000,0,0,0,0,2311,0,female,
3244688,20485,996220507785,6915367,B2C,Active,,YNGAILUU160WEEK,Ыңгайлуу 160,7 days,160.0,...,0,0.000000,0,0,0,0,1401,0,,
3244689,20485,996220507791,22592506,B2B_HYBRID,Suspended,,M2M100,M2M_100_25GB,1 months,100.0,...,0,0.000000,0,0,0,0,109,0,,
3244690,20485,996220507799,19972975,B2C,Terminated,,SUPERXMAX,Безлимит 500,30 days,500.0,...,0,0.000000,0,0,0,0,392,0,,


In [10]:
df_final.describe(include='all').T.to_csv(OUTPUT_DIR / f"ukg_final_eligible_{DATA_START}.csv", index=True)

In [16]:
OUTPUT_DIR

PosixPath('/data/aturov/universal_control_group/data/raw')

In [19]:
path_config.data_processed_path.mkdir(parents=True, exist_ok=True)

df_final.to_parquet(path_config.data_processed_path / f"ukg_eligible_final_{DATA_START}.parquet", index=False, compression="snappy")